# Compressed GLM demos

This notebook demonstrates the GLM estimators added to `duckreg`: logistic regression, Poisson regression, moderate-$K$ multinomial logit, and a many-label Poisson decomposition. The examples use synthetic data and a local DuckDB database.

In [1]:
import os, tempfile
import duckdb
import numpy as np
import pandas as pd
from duckreg.estimators import (
    DuckLogisticRegression,
    DuckPoissonRegression,
    DuckMultinomialLogisticRegression,
    DuckPoissonMultinomialRegression,
)

rng = np.random.default_rng(2026)
tmpdir = tempfile.mkdtemp()
db_path = os.path.join(tmpdir, "glm_demo.db")
print(db_path)

/var/folders/q6/krvpk33x13b1r8q4rm03p7240000gn/T/tmpp4ddduv6/glm_demo.db


## Logistic regression

The data are compressed by identical covariate patterns. For canonical logit, grouped sufficient statistics are count and sum of successes. The estimator below runs compressed IRLS; `method="one_step"` uses a subsample pilot plus one full-data Fisher step.

In [2]:
n = 20_000
x1 = rng.integers(0, 8, n)
x2 = rng.integers(0, 5, n)
p = 1 / (1 + np.exp(-(-1.0 + 0.3*x1 - 0.2*x2)))
y = rng.binomial(1, p)
df = pd.DataFrame({"y": y, "x1": x1, "x2": x2})
conn = duckdb.connect(db_path)
conn.register("df", df)
conn.execute("CREATE OR REPLACE TABLE binary_data AS SELECT * FROM df")
conn.close()

logit = DuckLogisticRegression(db_path, "binary_data", "y ~ x1 + x2", seed=1, method="irls")
logit.fit()
logit.fit_vcov()
print(pd.Series(logit.point_estimate, index=["Intercept", "x1", "x2"]))
print("compressed rows:", len(logit.df_compressed), "original rows:", n)

Intercept   -1.042109
x1           0.312891
x2          -0.196134
dtype: float64
compressed rows: 40 original rows: 20000


## Poisson regression

For log-link Poisson, the grouped sufficient statistics are count and sum of counts. The score is $\sum_g x_g(s_g-n_g\mu_g)$ and the information is $\sum_g n_g\mu_gx_gx_g'$.

In [3]:
mu = np.exp(0.4 + 0.12*x1 - 0.15*x2)
y_count = rng.poisson(mu)
df = pd.DataFrame({"y": y_count, "x1": x1, "x2": x2})
conn = duckdb.connect(db_path)
conn.register("df", df)
conn.execute("CREATE OR REPLACE TABLE count_data AS SELECT * FROM df")
conn.close()

pois = DuckPoissonRegression(db_path, "count_data", "y ~ x1 + x2", seed=1, method="irls")
pois.fit()
pois.fit_vcov()
print(pd.Series(pois.point_estimate, index=["Intercept", "x1", "x2"]))
print("compressed rows:", len(pois.df_compressed), "original rows:", n)

Intercept    0.385873
x1           0.122467
x2          -0.149246
dtype: float64
compressed rows: 40 original rows: 20000


## Multinomial logit for moderate label counts

For moderate $K$, `DuckMultinomialLogisticRegression` solves the joint baseline-category softmax problem on grouped class counts.

In [4]:
X = np.c_[np.ones(n), x1, x2]
beta = np.array([[0.1, 0.12, -0.15], [-0.3, -0.08, 0.25]])
eta = np.c_[X @ beta.T, np.zeros(n)]
prob = np.exp(eta - eta.max(axis=1, keepdims=True))
prob = prob / prob.sum(axis=1, keepdims=True)
y_class = np.array([rng.choice(["a", "b", "c"], p=row) for row in prob])
df = pd.DataFrame({"y": y_class, "x1": x1, "x2": x2})
conn = duckdb.connect(db_path)
conn.register("df", df)
conn.execute("CREATE OR REPLACE TABLE class_data AS SELECT * FROM df")
conn.close()

mn = DuckMultinomialLogisticRegression(db_path, "class_data", "y ~ x1 + x2", seed=1, baseline="c")
mn.fit()
print(pd.DataFrame(mn.point_estimate, index=mn.labels[:-1], columns=["Intercept", "x1", "x2"]))
print("compressed rows:", len(mn.df_compressed), "original rows:", n)

   Intercept        x1        x2
a   0.116738  0.111044 -0.133138
b  -0.269345 -0.085742  0.259998
compressed rows: 40 original rows: 20000


## Many-label Poisson decomposition

For many count labels, use the Taddy-style Poisson decomposition: fit many small label-wise Poisson regressions instead of a single huge softmax Hessian. This is natural for text/count-composition data.

In [5]:
labels = [f"token_{j:02d}" for j in range(12)]
rows = []
for i in range(600):
    xi1 = rng.integers(0, 6)
    xi2 = rng.integers(0, 4)
    for j, label in enumerate(labels):
        mu = np.exp(-1.5 + 0.03*j + 0.08*xi1 - 0.06*xi2)
        rows.append((i, label, rng.poisson(mu), xi1, xi2))
long = pd.DataFrame(rows, columns=["doc_id", "label", "count", "x1", "x2"])
conn = duckdb.connect(db_path)
conn.register("long", long)
conn.execute("CREATE OR REPLACE TABLE token_counts AS SELECT * FROM long")
conn.close()

pm = DuckPoissonMultinomialRegression(db_path, "token_counts", count_col="count", label_col="label", covars=["x1", "x2"], seed=1)
pm.fit()
print(pm.point_estimate.head())
print("labels fit:", len(pm.point_estimate), "long rows:", len(long), "compressed rows:", len(pm.df_compressed))

          Intercept        x1        x2
token_00  -2.154595  0.126040  0.158661
token_01  -1.508186  0.023240  0.000638
token_02  -1.096075  0.004880 -0.139319
token_03  -1.536776  0.068585 -0.027764
token_04  -1.244323  0.045169 -0.098990
labels fit: 12 long rows: 7200 compressed rows: 288
